# Agent 01 Quotes Realtime

Notebook operativo reducido.

Este notebook ya no lanza procesos. Su funcion es:
- fijar configuracion de prueba y de produccion,
- imprimir los comandos exactos para terminal,
- y visualizar artefactos/resultados del run.

La ejecucion real de Agent01/02/03 se hace desde terminal.

In [9]:
from pathlib import Path
import json
import pandas as pd

# =========================
# TEST RUN (el que se ha usado en terminal)
# test ejecutado para 3 tikers
#  - AEHL
#      - 2010-11-02 -> 2012-11-16
#      - 534 business days
#  - MDXG
#      - 2013-04-22 -> 2019-02-26
#      - 1527 business days
#  - INLF
#      - 2024-12-30 -> 2026-03-12
#      - 314 business days
# =========================
TEST_RUN_ID = "20260312_quotes_lifecycle3_test"
TEST_RUN_DIR = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit") / TEST_RUN_ID
TEST_QUOTES_ROOT = Path(r"D:\quotes\__pruebas__\lifecycle_prod")
TEST_TASKS_CSV = TEST_RUN_DIR / "inputs" / "tasks_lifecycle3.csv"
TEST_TASKS_META = TEST_RUN_DIR / "inputs" / "tasks_lifecycle3_meta.json"

# =========================
# PRODUCTION TEMPLATE
# =========================
PROD_RUN_ID = "YYYYMMDD_quotes_session_prod_01"
PROD_RUN_DIR = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit") / PROD_RUN_ID
PROD_QUOTES_ROOT = Path(r"D:\quotes")
PROD_TASKS_CSV = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\quotes_tasks_prod.csv")

DOWNLOADER_SCRIPT = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\scripts\download_quotes.py")
AGENT_RUNNERS_DIR = Path(globals().get("AGENT_RUNNERS_DIR", r"C:\TSIS_Data\v1\backtest_SmallCaps\scripts\agents"))
AGENT02_PS1 = Path(globals().get("AGENT02_PS1", AGENT_RUNNERS_DIR / "run_agent02_quotes_strict_loop.ps1"))
AGENT03_LIVE_PS1 = Path(globals().get("AGENT03_LIVE_PS1", AGENT_RUNNERS_DIR / "run_agent03_live_fast.ps1"))
AGENT03_COMPACT_PS1 = Path(globals().get("AGENT03_COMPACT_PS1", AGENT_RUNNERS_DIR / "run_agent03_monitor_compact.ps1"))

MAX_CONCURRENT = 24
TASK_BATCH_SIZE = 500
SESSION_START = "04:00:00"
SESSION_END = "20:00:00"

print("TEST_RUN_ID:", TEST_RUN_ID)
print("TEST_RUN_DIR:", TEST_RUN_DIR)
print("TEST_QUOTES_ROOT:", TEST_QUOTES_ROOT)
print("TEST_TASKS_CSV:", TEST_TASKS_CSV, "exists", TEST_TASKS_CSV.exists())
print("TEST_TASKS_META:", TEST_TASKS_META, "exists", TEST_TASKS_META.exists())
print("DOWNLOADER_SCRIPT exists:", DOWNLOADER_SCRIPT.exists())
print("AGENT02_PS1 exists:", AGENT02_PS1.exists())
print("AGENT03_LIVE_PS1 exists:", AGENT03_LIVE_PS1.exists())
print("AGENT03_COMPACT_PS1 exists:", AGENT03_COMPACT_PS1.exists())

TEST_RUN_ID: 20260312_quotes_lifecycle3_test
TEST_RUN_DIR: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test
TEST_QUOTES_ROOT: D:\quotes\__pruebas__\lifecycle_prod
TEST_TASKS_CSV: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\inputs\tasks_lifecycle3.csv exists True
TEST_TASKS_META: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\inputs\tasks_lifecycle3_meta.json exists True
DOWNLOADER_SCRIPT exists: True
AGENT02_PS1 exists: True
AGENT03_LIVE_PS1 exists: True
AGENT03_COMPACT_PS1 exists: True


## Comandos Reales de Prueba

Estos son los comandos que se han usado en terminal para la corrida de prueba.

Raiz de prueba:
- `D:\quotes\__pruebas__\lifecycle_prod`

Artefactos del run:
- `C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test`
- `C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\inputs\tasks_lifecycle3.csv`
- `C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\inputs\tasks_lifecycle3_meta.json`



```bash
python C:\TSIS_Data\v1\backtest_SmallCaps\scripts\download_quotes.py --csv C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\inputs\tasks_lifecycle3.csv --output D:\quotes\__pruebas__\lifecycle_prod --concurrent 24 --run-id 20260312_quotes_lifecycle3_test --run-dir C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test --task-batch-size 500 --session-start 04:00:00 --session-end 20:00:00
```

In [10]:
test_download_cmd = rf'''python {DOWNLOADER_SCRIPT} ^
  --csv "{TEST_TASKS_CSV}" ^
  --output "{TEST_QUOTES_ROOT}" ^
  --concurrent {MAX_CONCURRENT} ^
  --run-id "{TEST_RUN_ID}" ^
  --run-dir "{TEST_RUN_DIR}" ^
  --task-batch-size {TASK_BATCH_SIZE} ^
  --session-start {SESSION_START} ^
  --session-end {SESSION_END}'''

test_agent02_cmd = rf'''powershell -ExecutionPolicy Bypass -File "{AGENT02_PS1}" ^
  -RunId "{TEST_RUN_ID}" ^
  -QuotesRoot "{TEST_QUOTES_ROOT}" ^
  -MaxFiles 50000 ^
  -SleepSec 15 ^
  -ResetState'''

test_agent03_live_cmd = rf'''powershell -ExecutionPolicy Bypass -File "{AGENT03_LIVE_PS1}" ^
  -RunId "{TEST_RUN_ID}" ^
  -IntervalSec 10'''

test_agent03_compact_cmd = rf'''powershell -ExecutionPolicy Bypass -File "{AGENT03_COMPACT_PS1}" ^
  -RunId "{TEST_RUN_ID}" ^
  -SleepSec 30'''

print("=== TEST: AGENT01 ===")
print(test_download_cmd)
print()
print("=== TEST: AGENT02 ===")
print(test_agent02_cmd)
print()
print("=== TEST: AGENT03 LIVE ===")
print(test_agent03_live_cmd)
print()
print("=== TEST: AGENT03 COMPACT ===")
print(test_agent03_compact_cmd)

=== TEST: AGENT01 ===
python C:\TSIS_Data\v1\backtest_SmallCaps\scripts\download_quotes.py ^
  --csv "C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\inputs\tasks_lifecycle3.csv" ^
  --output "D:\quotes\__pruebas__\lifecycle_prod" ^
  --concurrent 24 ^
  --run-id "20260312_quotes_lifecycle3_test" ^
  --run-dir "C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test" ^
  --task-batch-size 500 ^
  --session-start 04:00:00 ^
  --session-end 20:00:00

=== TEST: AGENT02 ===
powershell -ExecutionPolicy Bypass -File "C:\Users\AlexJ\Desktop\tsis_agents\run_agent02_quotes_strict_loop.ps1" ^
  -RunId "20260312_quotes_lifecycle3_test" ^
  -QuotesRoot "D:\quotes\__pruebas__\lifecycle_prod" ^
  -MaxFiles 50000 ^
  -SleepSec 15 ^
  -ResetState

=== TEST: AGENT03 LIVE ===
powershell -ExecutionPolicy Bypass -File "C:\Users\AlexJ\Desktop\tsis_agents\run_agent03_live_fast.ps1" ^
  -RunId "20260312_quotes_lifecycle3_te

## Comandos Preparados de Produccion

Estos son templates. Ajusta `PROD_RUN_ID` y `PROD_TASKS_CSV` antes de lanzar.

In [11]:
prod_download_cmd = rf'''python {DOWNLOADER_SCRIPT} ^
  --csv "{PROD_TASKS_CSV}" ^
  --output "{PROD_QUOTES_ROOT}" ^
  --concurrent {MAX_CONCURRENT} ^
  --run-id "{PROD_RUN_ID}" ^
  --run-dir "{PROD_RUN_DIR}" ^
  --task-batch-size {TASK_BATCH_SIZE} ^
  --session-start {SESSION_START} ^
  --session-end {SESSION_END}'''

prod_agent02_cmd = rf'''powershell -ExecutionPolicy Bypass -File "{AGENT02_PS1}" ^
  -RunId "{PROD_RUN_ID}" ^
  -QuotesRoot "{PROD_QUOTES_ROOT}" ^
  -MaxFiles 50000 ^
  -SleepSec 15'''

prod_agent03_live_cmd = rf'''powershell -ExecutionPolicy Bypass -File "{AGENT03_LIVE_PS1}" ^
  -RunId "{PROD_RUN_ID}" ^
  -IntervalSec 10'''

prod_agent03_compact_cmd = rf'''powershell -ExecutionPolicy Bypass -File "{AGENT03_COMPACT_PS1}" ^
  -RunId "{PROD_RUN_ID}" ^
  -SleepSec 30'''

print("=== PROD: AGENT01 ===")
print(prod_download_cmd)
print()
print("=== PROD: AGENT02 ===")
print(prod_agent02_cmd)
print()
print("=== PROD: AGENT03 LIVE ===")
print(prod_agent03_live_cmd)
print()
print("=== PROD: AGENT03 COMPACT ===")
print(prod_agent03_compact_cmd)

=== PROD: AGENT01 ===
python C:\TSIS_Data\v1\backtest_SmallCaps\scripts\download_quotes.py ^
  --csv "C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\quotes_tasks_prod.csv" ^
  --output "D:\quotes" ^
  --concurrent 24 ^
  --run-id "YYYYMMDD_quotes_session_prod_01" ^
  --run-dir "C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\YYYYMMDD_quotes_session_prod_01" ^
  --task-batch-size 500 ^
  --session-start 04:00:00 ^
  --session-end 20:00:00

=== PROD: AGENT02 ===
powershell -ExecutionPolicy Bypass -File "C:\Users\AlexJ\Desktop\tsis_agents\run_agent02_quotes_strict_loop.ps1" ^
  -RunId "YYYYMMDD_quotes_session_prod_01" ^
  -QuotesRoot "D:\quotes" ^
  -MaxFiles 50000 ^
  -SleepSec 15

=== PROD: AGENT03 LIVE ===
powershell -ExecutionPolicy Bypass -File "C:\Users\AlexJ\Desktop\tsis_agents\run_agent03_live_fast.ps1" ^
  -RunId "YYYYMMDD_quotes_session_prod_01" ^
  -IntervalSec 10

=== PROD: AGENT03 COMPACT ===
powershell -ExecutionPolicy Bypass -File "C:\Users\AlexJ\Desktop\t

## Visualizacion de Artefactos del Run (resultados del test)

Esta parte sustituye al antiguo lanzamiento desde notebook.

Artefactos clave esperados:
- `quotes_agent_strict_events_current.csv`
- `retry_queue_quotes_strict_current.csv`
- `live_status_quotes_strict.json`
- `agent03_outputs/run_summary.json`
- `agent03_outputs/causes_by_ticker.csv`

In [12]:
events_current = TEST_RUN_DIR / "quotes_agent_strict_events_current.csv"
retry_current = TEST_RUN_DIR / "retry_queue_quotes_strict_current.csv"
live_status = TEST_RUN_DIR / "live_status_quotes_strict.json"
run_summary = TEST_RUN_DIR / "agent03_outputs" / "run_summary.json"
causes_by_ticker = TEST_RUN_DIR / "agent03_outputs" / "causes_by_ticker.csv"

for p in [events_current, retry_current, live_status, run_summary, causes_by_ticker]:
    print(p, "exists", p.exists())

if TEST_TASKS_META.exists():
    meta = json.loads(TEST_TASKS_META.read_text(encoding="utf-8"))
    print("\nTASKS META:")
    print(json.dumps(meta, indent=2, ensure_ascii=False))

if live_status.exists():
    print("\nLIVE STATUS:")
    print(json.dumps(json.loads(live_status.read_text(encoding="utf-8")), indent=2, ensure_ascii=False))

if events_current.exists():
    df = pd.read_csv(events_current)
    print("\nEVENTS CURRENT rows:", len(df))
    display(df.head(10))
    if "severity" in df.columns:
        print(df["severity"].value_counts(dropna=False))

if retry_current.exists():
    rq = pd.read_csv(retry_current)
    print("\nRETRY CURRENT rows:", len(rq))
    display(rq.head(10))

if run_summary.exists():
    print("\nRUN SUMMARY:")
    print(json.dumps(json.loads(run_summary.read_text(encoding="utf-8")), indent=2, ensure_ascii=False))

if causes_by_ticker.exists():
    cbt = pd.read_csv(causes_by_ticker)
    print("\nCAUSES BY TICKER rows:", len(cbt))
    display(cbt.head(20))

C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\quotes_agent_strict_events_current.csv exists True
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\retry_queue_quotes_strict_current.csv exists True
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\live_status_quotes_strict.json exists True
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\agent03_outputs\run_summary.json exists False
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\agent03_outputs\causes_by_ticker.csv exists False

TASKS META:
{
  "run_id": "20260312_quotes_lifecycle3_test",
  "summary": [
    {
      "era": "2005_2010",
      "ticker": "AEHL",
      "list_date": "2010-11-02",
      "end_date": "2012-11-16",
      "business_days": 534
    },
    {
      "era": "2011_2019",
      "ticker": "MDXG

,file,rows,severity,issues,warns,action,crossed_rows,crossed_ratio_pct,negative_price_rows,missing_required_cols,dtype_mismatches,ask_integer_pct,bid_integer_pct,ask_eq_round_bid_pct,processed_at_utc,run_id
0,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,1493,SOFT_FAIL,[],['crossed_rows_present_but_under_threshold'],warn_and_retry_later,7,0.468855,0,[],[],6.831882,4.621567,6.630944,2026-03-12 14:02:34.004431+00:00,20260312_quotes_lifecycle3_test
1,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,225,PASS,[],[],count_coverage,0,0.000000,0,[],[],2.666667,0.444444,2.666667,2026-03-12 14:02:34.006387+00:00,20260312_quotes_lifecycle3_test
2,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,1259,SOFT_FAIL,[],['crossed_rows_present_but_under_threshold'],warn_and_retry_later,2,0.158856,0,[],[],5.003971,1.906275,5.003971,2026-03-12 14:02:34.008231+00:00,20260312_quotes_lifecycle3_test
3,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,9700,SOFT_FAIL,[],['crossed_rows_present_but_under_threshold'],warn_and_retry_later,2,0.020619,0,[],[],1.536082,5.886598,1.536082,2026-03-12 14:02:34.010829+00:00,20260312_quotes_lifecycle3_test
4,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,260,PASS,[],[],count_coverage,0,0.000000,0,[],[],0.384615,0.769231,0.384615,2026-03-12 14:02:34.012418+00:00,20260312_quotes_lifecycle3_test
5,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,1601,PASS,[],[],count_coverage,0,0.000000,0,[],[],1.374141,2.186134,1.311680,2026-03-12 14:02:34.014075+00:00,20260312_quotes_lifecycle3_test
6,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,659,PASS,[],[],count_coverage,0,0.000000,0,[],[],7.435508,9.711684,7.283763,2026-03-12 14:02:34.015722+00:00,20260312_quotes_lifecycle3_test
7,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,399,PASS,[],[],count_coverage,0,0.000000,0,[],[],7.017544,2.756892,7.017544,2026-03-12 14:02:34.017265+00:00,20260312_quotes_lifecycle3_test
8,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,442,PASS,[],[],count_coverage,0,0.000000,0,[],[],0.452489,0.452489,0.226244,2026-03-12 14:02:34.018920+00:00,20260312_quotes_lifecycle3_test
9,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,273,PASS,[],[],count_coverage,0,0.000000,0,[],[],0.732601,1.098901,0.366300,2026-03-12 14:02:34.020412+00:00,20260312_quotes_lifecycle3_test


severity
SOFT_FAIL    222
PASS         176
HARD_FAIL      3
Name: count, dtype: int64

RETRY CURRENT rows: 0


,file,severity,issues,warns,action,processed_at_utc


In [4]:
from pathlib import Path
import pandas as pd
import json

run_id = "20260312_quotes_preprod30_full_lifecycle"
run_dir = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit") / run_id
inp = run_dir / "inputs"
inp.mkdir(parents=True, exist_ok=True)

universe = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet")
lifecycle = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\official_lifecycle_compiled.csv")

cutoff = pd.Timestamp("2026-03-12")

u = pd.read_parquet(universe)
lc = pd.read_csv(lifecycle)

tickers = sorted(set(u["ticker"].astype(str).str.upper()))
lc["ticker"] = lc["ticker"].astype(str).str.upper()

base = lc[lc["ticker"].isin(tickers)].copy()
base["list_date"] = pd.to_datetime(base["list_date"], errors="coerce")
base["delist_date"] = pd.to_datetime(base["delist_date"], errors="coerce")
base["end_date"] = base["delist_date"].fillna(cutoff)

base = base.dropna(subset=["ticker", "list_date", "end_date"]).copy()
base = base[base["end_date"] >= base["list_date"]].copy()

def pick_sample(df, y0, y1, n, seed):
    x = df[(df["list_date"].dt.year >= y0) & (df["list_date"].dt.year <= y1)].copy()
    if x.empty:
        return x
    return x.sample(n=min(n, len(x)), random_state=seed)

a = pick_sample(base, 2005, 2010, 10, 11)
b = pick_sample(base, 2011, 2020, 10, 22)
c = pick_sample(base, 2021, 2026, 10, 33)

sel = pd.concat([a, b, c], ignore_index=True).drop_duplicates(subset=["ticker"])
sel = sel.sort_values(["list_date", "ticker"]).reset_index(drop=True)

rows = []
for _, r in sel.iterrows():
    days = pd.bdate_range(r["list_date"].date(), r["end_date"].date())
    for d in days:
        rows.append({"ticker": r["ticker"], "date": d.strftime("%Y-%m-%d")})

tasks = pd.DataFrame(rows)

tasks_csv = inp / "tasks_preprod30_full_lifecycle.csv"
meta_json = inp / "tasks_preprod30_full_lifecycle_meta.json"
tickers_csv = inp / "tickers_preprod30_full_lifecycle.csv"

tasks.to_csv(tasks_csv, index=False)
sel.to_csv(tickers_csv, index=False)

meta = {
    "run_id": run_id,
    "universe_path": str(universe),
    "lifecycle_path": str(lifecycle),
    "tickers_selected": sel["ticker"].tolist(),
    "tickers_count": int(sel["ticker"].nunique()),
    "tasks_total": int(len(tasks)),
    "date_min": str(tasks["date"].min()) if len(tasks) else None,
    "date_max": str(tasks["date"].max()) if len(tasks) else None,
}
meta_json.write_text(json.dumps(meta, indent=2), encoding="utf-8")

print(tasks_csv)
print(meta_json)
print(tickers_csv)
print(meta)

C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_preprod30_full_lifecycle\inputs\tasks_preprod30_full_lifecycle.csv
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_preprod30_full_lifecycle\inputs\tasks_preprod30_full_lifecycle_meta.json
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_preprod30_full_lifecycle\inputs\tickers_preprod30_full_lifecycle.csv
{'run_id': '20260312_quotes_preprod30_full_lifecycle', 'universe_path': 'C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\universe_pti\\tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet', 'lifecycle_path': 'C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\official_lifecycle_compiled.csv', 'tickers_selected': ['DWSN', 'AVXL', 'IPM', 'ALLT', 'NOA', 'ADAM', 'NAGE', 'BPTH', 'VTGN', 'HRZN', 'FBIO', 'RGS', 'NAVI', 'OXBR', 'CSBR', 'PETZ', 'BV', 'IVF', 'ALIT', 'LXU', 'NRGV', 'FORA', 'DSGN', 'WT', 'JYD', 'VCIG',